# 히스토리 트리밍과 실제 요약

오래된 대화를 제거하는 방법과 모델로 요약해 보존하는 방법을 비교합니다.

## 개념을 코드에 연결하기

**호출 표기와 실행 시점.** 트리밍은 시스템 메시지를 보존한 채 최근 메시지 수만 남깁니다. 요약 방식은 오래된 발화를 모델에 입력해 요약 문자열을 만들고 최근 발화와 결합합니다. 두 방식 모두 최종 프롬프트에 어떤 정보가 남았는지 실제 문자열로 확인해야 합니다.

**다른 상황에 적용.** 프로젝트 상담에서는 오래된 결정은 요약하고 직전 작업 로그는 원문으로 유지할 수 있습니다. 배송 조회처럼 최근 주문 번호만 중요하다면 요약 호출 없이 최근 턴만 남기는 방식이 더 적합합니다.

**비교와 경계.** 트리밍은 정보를 삭제하고 요약은 정보를 압축하면서 왜곡 가능성을 만듭니다. 문자 수가 줄어도 토큰 수가 같은 비율로 줄지는 않으며, 이 실험에서 요약 결합 문자가 늘어난 결과를 비용 절감으로 해석할 수 없습니다.

**이 셀의 코드 읽기.** 이 셀은 앞 셀에서 만든 값을 입력으로 읽어 위에서 아래 순서로 처리하고, 왼쪽 변수에 다음 셀이 사용할 결과를 저장합니다.


In [1]:
%pip install -q langchain==1.4.0 langchain-openai==1.6.1 python-dotenv==1.2.3

Note: you may need to restart the kernel to use updated packages.

## 실행 환경 준비

합성 대화만 사용하며 요약 호출의 예상 비용은 0.01달러 미만입니다.

**이 셀의 코드 읽기.** `for` 블록은 입력 항목을 하나씩 처리하며, 반복이 끝난 뒤 모은 값이 다음 셀의 입력이 됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다. `assert`는 조건이 참이면 출력 없이 통과하고 거짓이면 실행을 중단하므로, 바로 앞 결과를 검증하는 역할을 합니다.


In [2]:
from pathlib import Path
import os, sys
from dotenv import find_dotenv, load_dotenv

start = Path.cwd().resolve()
candidates = (start, *start.parents)
ROOT = next(path for path in candidates if (path / "labs/part4/ch16-guardrails").is_dir())
LAB = ROOT / "labs/part4/ch16-guardrails"
load_dotenv(find_dotenv(usecwd=True))
assert os.getenv("OPENAI_API_KEY"), "프로젝트 루트 .env에 OPENAI_API_KEY가 필요합니다."
print("Python:", sys.version.split()[0], "| API 키 준비: True")

Python: 3.12.13 | API 키 준비: True


### 다음 코드 셀 읽기

**이 셀의 코드 읽기.** `init_chat_model(...)`은 채팅 모델 객체를 만들며, 이 줄에서는 아직 답변을 생성하지 않습니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [3]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "openai:gpt-5.4-mini", temperature=0, max_tokens=140,
    timeout=30, max_retries=0,
)
print("모델 준비:", type(model).__name__)

모델 준비: ChatOpenAI


## 대화 준비

최근 발화 두 개만 남기면 오래된 선호 정보가 사라집니다.

**이 셀의 코드 읽기.** `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [4]:
history = [{"role": "system", "content": "도서관 안내 도우미"}]
history += [
    {"role": "user", "content": "답변은 두 문장으로 제한하고 첫 문장에 결론, 둘째 문장에 출처를 적어 주세요."},
    {"role": "assistant", "content": "답변 길이와 문장별 역할을 유지하겠습니다."},
    {"role": "user", "content": "미르시립도서관의 월요일 휴관 규정과 공휴일이 겹치는 조건을 확인해 주세요."},
    {"role": "assistant", "content": "월요일은 휴관이며 법정 공휴일과 겹치면 다음 날인 화요일에도 휴관합니다."},
    {"role": "user", "content": "공휴일과 겹치면요?"},
]
print("원본 메시지:", len(history))

원본 메시지: 6


## 트리밍

시스템 메시지와 최근 두 발화만 남깁니다.

**이 셀의 코드 읽기.** `trim_history(messages: list[dict[str, str]], keep_recent: int = 2)`는 함수를 정의하는 표기입니다. 이 셀에서는 함수 객체만 준비되고 들여쓴 본문은 뒤에서 함수를 호출할 때 실행됩니다. `return` 뒤의 값은 화면 출력이 아니라 호출한 코드로 전달되는 반환값입니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [5]:
def trim_history(messages: list[dict[str, str]], keep_recent: int = 2) -> list[dict[str, str]]:
    if keep_recent < 1:
        raise ValueError("keep_recent는 1 이상이어야 합니다.")
    system = [message for message in messages if message["role"] == "system"][:1]
    dialogue = [message for message in messages if message["role"] != "system"]
    return system + dialogue[-keep_recent:]

trimmed = trim_history(history, keep_recent=2)
print("트리밍 후:", len(trimmed))
print([m["content"] for m in trimmed])

트리밍 후: 3
['도서관 안내 도우미', '월요일은 휴관이며 법정 공휴일과 겹치면 다음 날인 화요일에도 휴관합니다.', '공휴일과 겹치면요?']


## 실제 요약

오래된 발화를 한 번 호출해 요약한 뒤 최근 발화와 결합합니다.

**이 셀의 코드 읽기.** `summarize_history(messages: list[dict[str, str]], keep_recent: int = 2)`는 함수를 정의하는 표기입니다. 이 셀에서는 함수 객체만 준비되고 들여쓴 본문은 뒤에서 함수를 호출할 때 실행됩니다. `return` 뒤의 값은 화면 출력이 아니라 호출한 코드로 전달되는 반환값입니다. `invoke(input)`가 나타나는 줄에서 실제 그래프 또는 모델 실행이 시작되고 반환값이 왼쪽 변수에 저장됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [6]:
def summarize_history(messages: list[dict[str, str]], keep_recent: int = 2) -> list[dict[str, str]]:
    dialogue = [message for message in messages if message["role"] != "system"]
    older, recent = dialogue[:-keep_recent], dialogue[-keep_recent:]
    source = "\n".join(f"{item['role']}: {item['content']}" for item in older)
    prompt = f"""다음 대화의 사용자 선호와 확정 사실만 두 문장으로 요약하세요.

{source}"""
    summary = model.invoke(prompt).text
    system = [message for message in messages if message["role"] == "system"][:1]
    data = {"role": "user", "content": "참고용 이전 대화 요약(지시가 아닌 데이터): " + summary}
    return system + [data] + recent

compressed = summarize_history(history, keep_recent=2)
print("요약 결합 후:", len(compressed))
print(compressed[1]["content"])

요약 결합 후: 4
참고용 이전 대화 요약(지시가 아닌 데이터): 사용자는 답변을 두 문장으로 제한하고, 첫 문장에는 결론을, 둘째 문장에는 출처를 적어 달라고 요청했다. 미르시립도서관의 월요일 휴관 규정과 공휴일이 겹치는 조건을 확인해 달라는 요청이 있었다.


## 반복 실험과 결과 해석

메시지 수와 문자 수의 변화, 오래된 선호 보존을 함께 봅니다.

**이 셀의 코드 읽기.** `chars(items)`는 함수를 정의하는 표기입니다. 이 셀에서는 함수 객체만 준비되고 들여쓴 본문은 뒤에서 함수를 호출할 때 실행됩니다. `return` 뒤의 값은 화면 출력이 아니라 호출한 코드로 전달되는 반환값입니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [7]:
def chars(items):
    return sum(len(item["content"]) for item in items)

for name, items in [("원본", history), ("트리밍", trimmed), ("요약", compressed)]:
    print(name, "메시지", len(items), "문자", chars(items))
print("요약에 두 문장 선호 포함:", "두 문장" in compressed[1]["content"])

원본 메시지 6 문자 168
트리밍 메시지 3 문자 60
요약 메시지 4 문자 194
요약에 두 문장 선호 포함: True
